# Assignment 7: CUDA Part II


In [1]:
!nvcc --version && nvidia-smi

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0
Mon Apr 20 11:30:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8       

---
## Problem 1: Threads Performing Different Tasks

In [6]:
%%writefile problem1_thread_tasks.cu
// Problem 1: CUDA - Different Tasks per Thread
// Thread 0: Iterative sum of first N integers
// Thread 1: Formula-based sum of first N integers
// Compile: nvcc problem1_thread_tasks.cu -o problem1
// Run:     ./problem1

#include <cuda_runtime.h>
#include <stdio.h>
#include <stdlib.h>

#define N 1024   // Number of integers to sum

// ---------------------------------------------------------------------------
// Kernel: Each thread performs a DIFFERENT task based on its thread ID
//   Thread 0 → iterative sum: 1+2+3+...+N
//   Thread 1 → formula sum:   N*(N+1)/2
//   All other threads → identity (input[tid] copied to output[tid])
// ---------------------------------------------------------------------------
__global__ void differentTasks(int *input, long long *output, int n) {
    int tid = blockIdx.x * blockDim.x + threadIdx.x;

    if (tid == 0) {
        // Task A: Iterative sum (no direct formula)
        long long sum = 0;
        for (int i = 1; i <= n; i++) {
            sum += i;
        }
        output[0] = sum;

    } else if (tid == 1) {
        // Task B: Direct formula  N*(N+1)/2
        output[1] = (long long)n * (n + 1) / 2;

    } else if (tid < n) {
        // All other threads: copy their element (demonstrates parallel mapping)
        output[tid] = input[tid];
    }
}

int main() {
    printf("===== Problem 1: Different Tasks per Thread =====\n");
    printf("N = %d\n\n", N);

    // -----------------------------------------------------------------------
    // Step 1-2: Create input and output arrays on host
    // -----------------------------------------------------------------------
    int      *h_input  = (int*)      malloc(N * sizeof(int));
    long long *h_output = (long long*)malloc(N * sizeof(long long));

    // Step 4: Fill input with first N integers
    for (int i = 0; i < N; i++) h_input[i] = i + 1;

    // -----------------------------------------------------------------------
    // Step 3: Allocate device memory
    // -----------------------------------------------------------------------
    int      *d_input;
    long long *d_output;
    cudaMalloc(&d_input,  N * sizeof(int));
    cudaMalloc(&d_output, N * sizeof(long long));
    cudaMemset(d_output, 0, N * sizeof(long long));

    // Step 5: Copy host data to device
    cudaMemcpy(d_input, h_input, N * sizeof(int), cudaMemcpyHostToDevice);

    // -----------------------------------------------------------------------
    // Step 6: Define block and grid sizes
    // We use 1 block of N threads so every thread gets a unique tid 0..N-1
    // -----------------------------------------------------------------------
    dim3 block(N);   // 1024 threads per block
    dim3 grid(1);    // 1 block

    printf("Block size: %d threads\n", N);
    printf("Grid  size: 1 block\n\n");

    // -----------------------------------------------------------------------
    // Step 7: Launch kernel and retrieve results
    // -----------------------------------------------------------------------
    differentTasks<<<grid, block>>>(d_input, d_output, N);
    cudaDeviceSynchronize();

    cudaMemcpy(h_output, d_output, N * sizeof(long long), cudaMemcpyDeviceToHost);

    // Expected answer
    long long expected = (long long)N * (N + 1) / 2;

    printf("--- Task A (Thread 0): Iterative Sum ---\n");
    printf("  Result   : %lld\n", h_output[0]);
    printf("  Expected : %lld\n", expected);
    printf("  Match    : %s\n\n", h_output[0] == expected ? "YES" : "NO");

    printf("--- Task B (Thread 1): Formula Sum ---\n");
    printf("  Result   : %lld\n", h_output[1]);
    printf("  Expected : %lld\n", expected);
    printf("  Match    : %s\n\n", h_output[1] == expected ? "YES" : "NO");

    printf("Both threads agree: %s\n",
           h_output[0] == h_output[1] ? "YES" : "NO");

    // Cleanup
    cudaFree(d_input);
    cudaFree(d_output);
    free(h_input);
    free(h_output);
    return 0;
}

Overwriting problem1_thread_tasks.cu


In [7]:
!nvcc problem1_thread_tasks.cu -o problem1 && ./problem1

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
===== Problem 1: Different Tasks per Thread =====
N = 1024

Block size: 1024 threads
Grid  size: 1 block

--- Task A (Thread 0): Iterative Sum ---
  Result   : 524800
  Expected : 524800
  Match    : YES

--- Task B (Thread 1): Formula Sum ---
  Result   : 524800
  Expected : 524800
  Match    : YES

Both threads agree: YES


---
## Problem 2: Merge Sort — CPU Pipelined vs CUDA

In [8]:
%%writefile problem2_merge_sort.cu
// Problem 2: Merge Sort — Pipelined CPU vs CUDA Parallel
// Compile: nvcc problem2_merge_sort.cu -o problem2
// Run:     ./problem2

#include <cuda_runtime.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <time.h>

#define ARRAY_SIZE 1000

// ===========================================================================
// PART A: CPU Pipelined Merge Sort
// "Pipelining" here means the classic top-down recursive merge sort where
// successive merge passes naturally pipeline: while one level is merging
// pairs of sub-arrays, the previous level's output feeds it directly.
// ===========================================================================

void merge_cpu(int *arr, int *temp, int left, int mid, int right) {
    int i = left, j = mid + 1, k = left;
    while (i <= mid && j <= right)
        temp[k++] = (arr[i] <= arr[j]) ? arr[i++] : arr[j++];
    while (i <= mid)  temp[k++] = arr[i++];
    while (j <= right) temp[k++] = arr[j++];
    for (int idx = left; idx <= right; idx++) arr[idx] = temp[idx];
}

void mergeSort_cpu(int *arr, int *temp, int left, int right) {
    if (left >= right) return;
    int mid = left + (right - left) / 2;
    mergeSort_cpu(arr, temp, left, mid);       // Pipeline stage 1: sort left
    mergeSort_cpu(arr, temp, mid + 1, right);  // Pipeline stage 2: sort right
    merge_cpu(arr, temp, left, mid, right);    // Pipeline stage 3: merge
}

// ===========================================================================
// PART B: CUDA Parallel Merge Sort
//
// Strategy: Iterative bottom-up merge sort.
// Pass 1: Each thread merges pairs of size-1 sub-arrays (1024 parallel merges)
// Pass 2: Each thread merges pairs of size-2 sub-arrays (512 parallel merges)
// ...continuing until the array is fully sorted.
// This is embarrassingly parallel at each level.
// ===========================================================================

__device__ void merge_gpu(int *arr, int *temp, int left, int mid, int right) {
    int i = left, j = mid + 1, k = left;
    while (i <= mid && j <= right)
        temp[k++] = (arr[i] <= arr[j]) ? arr[i++] : arr[j++];
    while (i <= mid)   temp[k++] = arr[i++];
    while (j <= right) temp[k++] = arr[j++];
    for (int idx = left; idx <= right; idx++) arr[idx] = temp[idx];
}

// Each thread handles one merge of a sub-array pair of size `width`
__global__ void mergeSortKernel(int *arr, int *temp, int n, int width) {
    int tid  = blockIdx.x * blockDim.x + threadIdx.x;
    int left = tid * 2 * width;

    if (left >= n) return;

    int mid   = min(left + width - 1, n - 1);
    int right = min(left + 2 * width - 1, n - 1);

    if (mid < right)
        merge_gpu(arr, temp, left, mid, right);
}

// ===========================================================================
// Helper utilities
// ===========================================================================

void fill_random(int *arr, int n, unsigned int seed) {
    srand(seed);
    for (int i = 0; i < n; i++) arr[i] = rand() % 10000;
}

int is_sorted(int *arr, int n) {
    for (int i = 0; i < n - 1; i++)
        if (arr[i] > arr[i + 1]) return 0;
    return 1;
}

double get_time_ms(struct timespec start, struct timespec end) {
    return (end.tv_sec - start.tv_sec) * 1000.0
         + (end.tv_nsec - start.tv_nsec) / 1e6;
}

// ===========================================================================
// Main
// ===========================================================================

int main() {
    printf("===== Problem 2: Merge Sort — CPU Pipeline vs CUDA =====\n");
    printf("Array size: %d\n\n", ARRAY_SIZE);

    int *arr_cpu  = (int*)malloc(ARRAY_SIZE * sizeof(int));
    int *temp_cpu = (int*)malloc(ARRAY_SIZE * sizeof(int));
    int *arr_gpu  = (int*)malloc(ARRAY_SIZE * sizeof(int));

    // -----------------------------------------------------------------------
    // Part A: CPU Pipelined Merge Sort
    // -----------------------------------------------------------------------
    fill_random(arr_cpu, ARRAY_SIZE, 42);

    struct timespec t0, t1;
    clock_gettime(CLOCK_MONOTONIC, &t0);
    mergeSort_cpu(arr_cpu, temp_cpu, 0, ARRAY_SIZE - 1);
    clock_gettime(CLOCK_MONOTONIC, &t1);

    double cpu_time = get_time_ms(t0, t1);
    printf("--- Part A: CPU Pipelined Merge Sort ---\n");
    printf("  Time      : %.4f ms\n", cpu_time);
    printf("  Sorted    : %s\n\n", is_sorted(arr_cpu, ARRAY_SIZE) ? "YES" : "NO");

    // -----------------------------------------------------------------------
    // Part B: CUDA Parallel Merge Sort
    // -----------------------------------------------------------------------
    fill_random(arr_gpu, ARRAY_SIZE, 42);   // same seed → same input

    int *d_arr, *d_temp;
    cudaMalloc(&d_arr,  ARRAY_SIZE * sizeof(int));
    cudaMalloc(&d_temp, ARRAY_SIZE * sizeof(int));
    cudaMemcpy(d_arr, arr_gpu, ARRAY_SIZE * sizeof(int), cudaMemcpyHostToDevice);

    // CUDA timing
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    int block_size = 128;
    cudaEventRecord(start);

    // Iterative bottom-up: width doubles each pass
    for (int width = 1; width < ARRAY_SIZE; width *= 2) {
        int merges   = (ARRAY_SIZE + 2 * width - 1) / (2 * width);
        int blocks   = (merges + block_size - 1) / block_size;
        mergeSortKernel<<<blocks, block_size>>>(d_arr, d_temp, ARRAY_SIZE, width);
        cudaDeviceSynchronize();
    }

    cudaEventRecord(stop);
    cudaEventSynchronize(stop);
    float gpu_time = 0;
    cudaEventElapsedTime(&gpu_time, start, stop);

    cudaMemcpy(arr_gpu, d_arr, ARRAY_SIZE * sizeof(int), cudaMemcpyDeviceToHost);

    printf("--- Part B: CUDA Parallel Merge Sort ---\n");
    printf("  Time      : %.4f ms\n", gpu_time);
    printf("  Sorted    : %s\n\n", is_sorted(arr_gpu, ARRAY_SIZE) ? "YES" : "NO");

    // -----------------------------------------------------------------------
    // Part C: Performance Comparison
    // -----------------------------------------------------------------------
    printf("--- Part C: Performance Comparison ---\n");
    printf("  CPU time  : %.4f ms\n", cpu_time);
    printf("  GPU time  : %.4f ms\n", gpu_time);
    if (gpu_time < cpu_time)
        printf("  Speedup   : %.2fx  (GPU faster)\n", cpu_time / gpu_time);
    else
        printf("  Slowdown  : %.2fx  (CPU faster — small N favors CPU due to kernel launch overhead)\n",
               gpu_time / cpu_time);
    printf("\n  NOTE: For n=1000, CPU overhead from memory transfer and\n");
    printf("  kernel launches often outweighs GPU compute gains.\n");
    printf("  GPU advantage becomes decisive at n >= 100,000+.\n");

    // Verify both produce the same result
    int match = 1;
    for (int i = 0; i < ARRAY_SIZE; i++)
        if (arr_cpu[i] != arr_gpu[i]) { match = 0; break; }
    printf("\n  CPU and GPU outputs match: %s\n", match ? "YES" : "NO");

    cudaFree(d_arr); cudaFree(d_temp);
    free(arr_cpu); free(temp_cpu); free(arr_gpu);
    cudaEventDestroy(start); cudaEventDestroy(stop);
    return 0;
}

Overwriting problem2_merge_sort.cu


In [9]:
!nvcc problem2_merge_sort.cu -o problem2 && ./problem2

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
===== Problem 2: Merge Sort — CPU Pipeline vs CUDA =====
Array size: 1000

--- Part A: CPU Pipelined Merge Sort ---
  Time      : 0.1535 ms
  Sorted    : YES

--- Part B: CUDA Parallel Merge Sort ---
  Time      : 30.3043 ms
  Sorted    : YES

--- Part C: Performance Comparison ---
  CPU time  : 0.1535 ms
  GPU time  : 30.3043 ms
  Slowdown  : 197.42x  (CPU faster — small N favors CPU due to kernel launch overhead)

  NOTE: For n=1000, CPU overhead from memory transfer and
  kernel launches often outweighs GPU compute gains.
  GPU advantage becomes decisive at n >= 100,000+.

  CPU and GPU outputs match: YES


---
## Problem 3: Vector Addition — Static Variables, Timing, Bandwidth

In [15]:
%%writefile problem3_vector_add.cu
#include <cuda_runtime.h>
#include <stdio.h>
#include <stdlib.h>
#include <math.h>

#define VECTOR_SIZE (1 << 20)
#define BLOCK_SIZE  256

__global__ void vectorAdd(float *A, float *B, float *C, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) C[i] = A[i] + B[i];
}

int main() {
    printf("===== Problem 3: Vector Addition -- Bandwidth Analysis =====\n");

    int n = VECTOR_SIZE;
    size_t bytes = n * sizeof(float);

    float *h_A = (float*)malloc(bytes);
    float *h_B = (float*)malloc(bytes);
    float *h_C = (float*)malloc(bytes);
    for (int i = 0; i < n; i++) { h_A[i] = i * 0.5f; h_B[i] = i * 1.5f; }

    float *d_A, *d_B, *d_C;
    cudaMalloc(&d_A, bytes);
    cudaMalloc(&d_B, bytes);
    cudaMalloc(&d_C, bytes);

    cudaMemcpy(d_A, h_A, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, bytes, cudaMemcpyHostToDevice);

    int gridSize = (n + BLOCK_SIZE - 1) / BLOCK_SIZE;
    printf("Vector size : %d elements\n", n);
    printf("Block size  : %d threads\n", BLOCK_SIZE);
    printf("Grid  size  : %d blocks\n\n", gridSize);

    /* 1.3 Theoretical bandwidth */
    cudaDeviceProp prop;
    cudaGetDeviceProperties(&prop, 0);
    printf("--- Device Properties ---\n");
    printf("  GPU                  : %s\n", prop.name);
    printf("  Memory Clock Rate    : %d kHz\n", prop.memoryClockRate);
    printf("  Memory Bus Width     : %d bits\n", prop.memoryBusWidth);

    double theoreticalBW = 2.0 * prop.memoryClockRate * 1000.0
                           * (prop.memoryBusWidth / 8.0) / 1.0e9;
    printf("\n--- 1.3 Theoretical Bandwidth ---\n");
    printf("  Formula : 2 x memClockRate(Hz) x busWidth(bytes) / 1e9\n");
    printf("  Theoretical BW : %.2f GB/s\n\n", theoreticalBW);

    /* 1.2 Kernel timing */
    cudaEvent_t evStart, evStop;
    cudaEventCreate(&evStart);
    cudaEventCreate(&evStop);

    cudaEventRecord(evStart);
    vectorAdd<<<gridSize, BLOCK_SIZE>>>(d_A, d_B, d_C, n);
    cudaEventRecord(evStop);
    cudaEventSynchronize(evStop);

    float kernel_ms = 0.0f;
    cudaEventElapsedTime(&kernel_ms, evStart, evStop);
    printf("--- 1.2 Kernel Timing ---\n");
    printf("  Kernel execution time : %.4f ms\n\n", kernel_ms);

    /* 1.4 Measured bandwidth */
    long long RBytes    = (long long)n * 2 * sizeof(float);
    long long WBytes    = (long long)n * 1 * sizeof(float);
    long long totalBytes = RBytes + WBytes;
    double measuredBW   = (double)totalBytes / (kernel_ms / 1000.0) / 1.0e9;

    printf("--- 1.4 Measured Bandwidth ---\n");
    printf("  Bytes read     : %lld bytes (%.2f MB)\n", RBytes,  RBytes  / 1.0e6);
    printf("  Bytes written  : %lld bytes (%.2f MB)\n", WBytes,  WBytes  / 1.0e6);
    printf("  Total bytes    : %lld bytes (%.2f MB)\n", totalBytes, totalBytes / 1.0e6);
    printf("  Kernel time    : %.6f s\n", kernel_ms / 1000.0);
    printf("  Measured BW    : %.2f GB/s\n\n", measuredBW);

    printf("========================================\n");
    printf("  Theoretical BW : %8.2f GB/s\n", theoreticalBW);
    printf("  Measured BW    : %8.2f GB/s\n", measuredBW);
    printf("  Efficiency     : %8.2f %%\n", (measuredBW / theoreticalBW) * 100.0);
    printf("========================================\n\n");

    /* Verify */
    cudaMemcpy(h_C, d_C, bytes, cudaMemcpyDeviceToHost);
    int errors = 0;
    for (int i = 0; i < n; i++)
        if (fabsf(h_C[i] - (h_A[i] + h_B[i])) > 1e-4f) errors++;
    printf("Verification : %s  (errors: %d)\n", errors == 0 ? "PASS" : "FAIL", errors);

    cudaFree(d_A); cudaFree(d_B); cudaFree(d_C);
    free(h_A); free(h_B); free(h_C);
    cudaEventDestroy(evStart); cudaEventDestroy(evStop);
    return 0;
}

Overwriting problem3_vector_add.cu


In [16]:
!nvcc -O3 problem3_vector_add.cu -o problem3 && ./problem3

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
===== Problem 3: Vector Addition -- Bandwidth Analysis =====
Vector size : 1048576 elements
Block size  : 256 threads
Grid  size  : 4096 blocks

--- Device Properties ---
  GPU                  : Tesla T4
  Memory Clock Rate    : 5001000 kHz
  Memory Bus Width     : 256 bits

--- 1.3 Theoretical Bandwidth ---
  Formula : 2 x memClockRate(Hz) x busWidth(bytes) / 1e9
  Theoretical BW : 320.06 GB/s

--- 1.2 Kernel Timing ---
  Kernel execution time : 12.4378 ms

--- 1.4 Measured Bandwidth ---
  Bytes read     : 8388608 bytes (8.39 MB)
  Bytes written  : 4194304 bytes (4.19 MB)
  Total bytes    : 12582912 bytes (12.58 MB)
  Kernel time    : 0.012438 s
  Measured BW    : 1.01 GB/s

  Theoretical BW :   320.06 GB/s
  Measured BW    :     1.01 GB/s
  Efficiency     :     0.32 %

Verification : PASS  (errors: 

In [12]:
!nvprof ./problem3 2>&1 | head -60

======== Error: Application received signal 139
